# Homework 03 - AI Orchestration with Kestra Answers

Notebook de respuestas para la Homework 3 del LLM Zoomcamp 2026.

Tema del modulo: orquestacion de workflows de IA con Kestra, comparando prompts simples, RAG, agentes, uso autonomo de herramientas y sistemas multi-agente.

La idea importante de este modulo es separar dos cosas:

- **Workflow deterministico**: pasos definidos explicitamente en YAML, utiles cuando necesito repetibilidad, auditoria y control.
- **Agente**: un LLM con instrucciones y herramientas, capaz de decidir que acciones tomar segun el contexto.


## Final Answers

| Question | Answer |
|---|---|
| Q1 | `b) Copilot has access to current Kestra plugin documentation` |
| Q2 | `a) RAG version provides specific, accurate feature details grounded in the documentation` |
| Q3 | `b) Long summary uses 2-4x more output tokens than short summary` |
| Q4 | `b) The agent autonomously decides based on the prompt and system message` |
| Q5 | `b) It serves as a tool for the main agent to gather web data` |
| Q6 | `b) Use traditional task-based workflows for predictability and auditability` |


## Setup Context

La homework asume Kestra corriendo localmente y los flows del modulo importados.

Conceptualmente, Kestra actua como orquestador: define inputs, tareas, dependencias, ejecuciones y logs. En workflows de IA, eso permite observar prompts, respuestas, token usage, llamadas a herramientas y resultados intermedios.

Los secretos se pasan como variables de entorno codificadas en base64, por ejemplo:

```bash
export SECRET_GEMINI_API_KEY=$(echo -n "your-gemini-api-key" | base64)
export SECRET_TAVILY_API_KEY=$(echo -n "your-tavily-api-key" | base64)
```

En los flows se leen como:

```text
{{ secret('GEMINI_API_KEY') }}
```

Sin el prefijo `SECRET_` dentro de `secret()`.

## Q1 - Context Engineering

**Question:** After trying the same prompt in ChatGPT vs Kestra's AI Copilot, what is the primary reason Copilot generates better Kestra flows?

**Answer:** `b) Copilot has access to current Kestra plugin documentation`

### Reasoning

Un modelo generico puede conocer ideas generales de YAML, BigQuery o pipelines, pero no necesariamente conoce el schema exacto y actualizado de los plugins de Kestra.

Kestra Copilot esta mejor contextualizado para generar flows validos porque tiene acceso al contexto del producto y a documentacion actual de plugins. Esto es context engineering: no se trata solamente de usar un modelo mas grande, sino de darle el contexto correcto para la tarea concreta.

### Concept

Context engineering significa disenar que informacion recibe el modelo para que sus respuestas sean utiles, validas y especificas del dominio.

## Q2 - RAG Comparison

**Question:** Run both `1_chat_without_rag.yaml` and `2_chat_with_rag.yaml`. Both ask: `Which features were released in Kestra 1.1?` What difference do you observe?

**Answer:** `a) RAG version provides specific, accurate feature details grounded in the documentation`

### Reasoning

Sin RAG, el modelo responde usando conocimiento interno y puede quedar desactualizado o inventar detalles. Con RAG, el flow recupera documentacion relevante y la agrega como contexto antes de responder.

La version con RAG deberia producir una respuesta mas concreta y trazable porque esta grounded in documentation.

### Concept

RAG reduce hallucinations porque la respuesta se apoya en documentos recuperados, no solo en memoria del modelo.

## Q3 - Token Usage

**Question:** Run `3_simple_agent.yaml` twice: first with `summary_length = short`, second with `summary_length = long`. How does token usage differ for the `multilingual_agent` task?

**Answer:** `b) Long summary uses 2-4x more output tokens than short summary`

### Reasoning

La diferencia principal esta en output tokens. Una respuesta larga obliga al modelo a generar mas texto, por lo tanto consume mas tokens de salida.

Los input tokens pueden ser parecidos porque el prompt base y el contexto son similares; lo que cambia fuerte es la longitud pedida para la respuesta.

### Concept

En sistemas con LLMs, el costo no depende solo del modelo. Tambien depende de:

- cantidad de contexto enviado,
- cantidad de herramientas usadas,
- longitud de la respuesta generada,
- numero de iteraciones del agente.

## Q4 - Agent Autonomy

**Question:** In `4_web_research_agent.yaml`, who decides when to use the web search tool?

**Answer:** `b) The agent autonomously decides based on the prompt and system message`

### Reasoning

En un workflow tradicional, el disenador define el orden exacto de tareas: primero A, despues B, despues C.

En un agente con tools, el workflow le da al modelo una herramienta disponible, pero el modelo decide si la necesita y cuando llamarla. Esa decision depende de las instrucciones, del system prompt, de la pregunta y del estado de la conversacion.

### Concept

La autonomia del agente aparece cuando el LLM puede elegir acciones. Eso lo hace flexible, pero tambien menos deterministico que un workflow tradicional.

## Q5 - Multi-Agent Collaboration

**Question:** In `5_multi_agent_research.yaml`, what is the role of the research agent in this multi-agent system?

**Answer:** `b) It serves as a tool for the main agent to gather web data`

### Reasoning

En un sistema multi-agente, no todos los agentes tienen el mismo rol. Normalmente hay un agente principal que coordina el trabajo y agentes especializados que resuelven subtareas.

En este caso, el research agent se usa como una herramienta especializada para recolectar informacion web. El main agent puede usar esos resultados para estructurar el analisis final.

### Concept

Un multi-agent system delega partes del problema. La ventaja es modularidad: cada agente puede tener instrucciones, herramientas y responsabilidades distintas.

## Q6 - Best Practices

**Question:** For production workflows requiring deterministic, repeatable results with strict compliance requirements, which approach is most appropriate?

**Answer:** `b) Use traditional task-based workflows for predictability and auditability`

### Reasoning

Los agentes son utiles cuando la tarea requiere exploracion, adaptacion o uso flexible de herramientas. Pero esa flexibilidad trae variabilidad.

Para industrias reguladas, reporting financiero o procesos con auditoria estricta, suele importar mas la repetibilidad que la autonomia. Un workflow tradicional permite saber exactamente que paso se ejecuto, con que input, en que orden y con que resultado.

### Concept

Regla practica:

- Si necesito exploracion y adaptacion: agente.
- Si necesito control, auditoria y repetibilidad: workflow deterministico.
- Si necesito respuestas grounded en documentos: RAG.
- Si necesito informacion actualizada externa: tool/web search, con controles.

## Mental Model of Module 3

Este modulo no se trata solamente de usar Kestra. Se trata de entender como llevar aplicaciones con LLMs a workflows observables y controlables.

### 1. Plain LLM call

```text
prompt -> LLM -> answer
```

Simple, pero puede inventar o quedar desactualizado.

### 2. RAG workflow

```text
question -> retrieve docs -> prompt with context -> LLM -> grounded answer
```

Mejor para responder con informacion documentada.

### 3. Agent workflow

```text
goal -> LLM decides action -> tool call -> observation -> repeat -> final answer
```

Mejor para tareas donde el camino no esta completamente definido.

### 4. Multi-agent workflow

```text
main agent -> delegates to specialized agent/tool -> collects result -> final synthesis
```

Mejor para problemas complejos que conviene dividir en roles.

## Submission Dictionary

In [1]:
answers = {
    "Q1": "b",
    "Q2": "a",
    "Q3": "b",
    "Q4": "b",
    "Q5": "b",
    "Q6": "b",
}

answers

{'Q1': 'b', 'Q2': 'a', 'Q3': 'b', 'Q4': 'b', 'Q5': 'b', 'Q6': 'b'}